# TV3 — Tuần 2: Huấn luyện 6 Cấu hình Thực nghiệm `M2: 3D Attention U-Net`

Notebook này thực thi trọn vẹn công việc **Tuần 2 của TV3** theo `docs/task_assignment.md` và `docs/detailed_pipeline.md`, bao gồm **6 cấu hình (`Run 07` $\rightarrow$ `Run 12`)**:

| Run ID | Tên thư mục Checkpoint | Kiến trúc | Đầu vào (`ip_mode`) | Số kênh | Điều kiện hóa Subtype (`FiLM`) |
| :--- | :--- | :--- | :--- | :---: | :--- |
| **Run 07** | `run07_M2_IP1_post1_no_film` | `M2: 3D Attention U-Net` | `IP1_post1` ($C_1$: Post-contrast) | 1 | Không (`use_film=False`) |
| **Run 08** | `run08_M2_IP1_post1_film` | `M2: 3D Attention U-Net` | `IP1_post1` ($C_1$: Post-contrast) | 1 | Có (`MultiStageFiLM`) |
| **Run 09** | `run09_M2_IP2_sub_no_film` | `M2: 3D Attention U-Net` | `IP2_sub` ($C_2$: Subtraction) | 1 | Không (`use_film=False`) |
| **Run 10** | `run10_M2_IP2_sub_film` | `M2: 3D Attention U-Net` | `IP2_sub` ($C_2$: Subtraction) | 1 | Có (`MultiStageFiLM`) |
| **Run 11** | `run11_M2_IP3_multi_no_film` | `M2: 3D Attention U-Net` | `IP3_multi` ($C_1 + C_2 + C_3$) | 3 | Không (`use_film=False`) |
| **Run 12** | `run12_M2_IP3_multi_film` | `M2: 3D Attention U-Net` | `IP3_multi` ($C_1 + C_2 + C_3$) | 3 | Có (`MultiStageFiLM`) |

### Cấu trúc thư mục trên Google Drive (`CS406-IPA/`)
Thư mục trên Google Drive của bạn (`/content/drive/My Drive/CS406-IPA/`) có cấu trúc **giống hệt** thư mục `SCB-TumorSeg/` dưới máy tính cá nhân:
- `configs/default_config.yaml`
- `src/` (`datasets/`, `models/`, `losses/`, `engine/`, `utils/`, `preprocessing/`)
- `scripts/train_tv3_week2.py`
- `notebooks/TV3_Week2_AttentionUNet_6Runs.ipynb`
- `data/splits/splits_metadata.csv` & `data/processed/npz_96x96x96/` (1.480 files `.npz`)
- `outputs/` $\rightarrow$ Nơi lưu tự động `checkpoints/tv3_week2/`, `tables/tv3_week2_ablation_summary.csv` và `figures/tv3_week2_learning_curves.png`. Khi chạy xong, bạn chỉ cần tải thư mục `outputs/` về đặt vào đúng `SCB-TumorSeg/outputs/` dưới máy.

In [ ]:
# 1. Mount Google Drive, chuyển vào thư mục CS406-IPA và cài đặt thư viện cần thiết
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/My Drive/CS406-IPA")
    os.chdir(PROJECT_ROOT)
    %pip install -q monai nibabel SimpleITK scikit-image pyyaml tqdm pandas matplotlib
else:
    PROJECT_ROOT = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/models/factory.py").is_file()),
        Path.cwd(),
    )
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working Directory (PROJECT_ROOT):", PROJECT_ROOT)

In [ ]:
# 2. [TỐI ƯU TỐC ĐỘ COLAB] Copy 1.480 file .npz từ Google Drive sang ổ SSD cục bộ của máy ảo Colab
# Đọc 1.480 file qua mạng Google Drive FUSE mỗi epoch sẽ rất chậm. Copy 1 lần sang /content/local_npz_96x96x96
# chỉ mất ~1-2 phút nhưng giúp tốc độ train nhanh gấp ~10 lần. Kết quả outputs/ vẫn lưu thẳng về Google Drive!
import shutil
from concurrent.futures import ThreadPoolExecutor

DRIVE_NPZ_DIR = PROJECT_ROOT / "data" / "processed" / "npz_96x96x96"
METADATA_CSV = PROJECT_ROOT / "data" / "splits" / "splits_metadata.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

USE_LOCAL_SSD_CACHE = True  # Đặt True trên Colab để copy .npz sang SSD cục bộ /content/

if IN_COLAB and USE_LOCAL_SSD_CACHE:
    LOCAL_NPZ_DIR = Path("/content/local_npz_96x96x96")
    LOCAL_NPZ_DIR.mkdir(parents=True, exist_ok=True)
    src_files = sorted(DRIVE_NPZ_DIR.glob("*.npz"))
    missing_files = [f for f in src_files if not (LOCAL_NPZ_DIR / f.name).is_file()]
    if missing_files:
        print(f"Đang copy {len(missing_files)}/{len(src_files)} file .npz từ Drive sang SSD cục bộ Colab...")
        def _copy_one(src_path: Path):
            shutil.copy2(src_path, LOCAL_NPZ_DIR / src_path.name)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(_copy_one, missing_files))
    NPZ_DIR = LOCAL_NPZ_DIR
    print(f"Hoàn tất! Đang sử dụng {len(list(NPZ_DIR.glob('*.npz')))} file .npz tại SSD cục bộ: {NPZ_DIR}")
else:
    NPZ_DIR = DRIVE_NPZ_DIR
    print(f"Đang sử dụng {len(list(NPZ_DIR.glob('*.npz')))} file .npz trực tiếp tại: {NPZ_DIR}")

In [ ]:
# 3. Kiểm tra phần cứng GPU & Hợp đồng dữ liệu trước khi chạy
import pandas as pd
import torch
import monai

print("PyTorch:", torch.__version__, "| MONAI:", monai.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

df_meta = pd.read_csv(METADATA_CSV)
print(f"\nTổng số ca trong splits_metadata.csv: {len(df_meta)}")
print(df_meta.groupby(["split", "tumor_subtype"]).size().unstack(fill_value=0))

In [ ]:
# 4. Cấu hình tham số huấn luyện trên GPU A100
SELECTED_RUNS = ["all"]

EPOCHS = 120
WARMUP_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 25
SAVE_LAST_EVERY = 20

BATCH_SIZE = 16
SW_BATCH_SIZE = 8
SW_OVERLAP = 0.5

LR = 3e-4
WEIGHT_DECAY = 1e-5
USE_AMP = True
NUM_WORKERS = 12
CACHE_RATE = 0.0
RESUME = True
SEED = 42

print(f"Config: BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={EARLY_STOPPING_PATIENCE}, USE_AMP={USE_AMP}")

In [ ]:
# 5. Thực thi huấn luyện 6 cấu hình M2 (3D Attention U-Net)
from scripts.train_tv3_week2 import run_tv3_week2_suite

summary_df = run_tv3_week2_suite(
    metadata_csv=METADATA_CSV,
    npz_dir=NPZ_DIR,
    output_root=OUTPUT_ROOT,
    config_path=PROJECT_ROOT / "configs" / "default_config.yaml",
    selected_runs=SELECTED_RUNS,
    epochs=EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patch_size=(96, 96, 96),
    sw_batch_size=SW_BATCH_SIZE,
    sw_overlap=SW_OVERLAP,
    num_workers=NUM_WORKERS,
    cache_rate=CACHE_RATE,
    use_amp=USE_AMP,
    use_weighted_sampler=True,
    save_last_every=SAVE_LAST_EVERY,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    resume=RESUME,
    seed=SEED,
)

In [ ]:
# 6. Hiển thị bảng tổng hợp kết quả Ablation và biểu đồ Learning Curves của TV3
from IPython.display import Image, display

summary_csv = OUTPUT_ROOT / "tables" / "tv3_week2_ablation_summary.csv"
curve_png = OUTPUT_ROOT / "figures" / "tv3_week2_learning_curves.png"

if summary_csv.is_file():
    df_res = pd.read_csv(summary_csv)
    display(df_res)

if curve_png.is_file():
    display(Image(filename=str(curve_png)))

In [ ]:
# 7. Drive Sync & Verification Lock (Khóa an toàn: đảm bảo 100% dữ liệu đã ghi xuống Drive)
import os
import time
import torch

print("Đang đồng bộ dữ liệu từ bộ đệm hệ thống xuống Google Drive...")
try:
    os.sync()
except Exception:
    pass
# Đợi 10 giây để daemon Google Drive FUSE đẩy toàn bộ khối dữ liệu lên cloud
time.sleep(10)

# Khóa kiểm định: Đọc thử toàn bộ file checkpoint đã tạo để đảm bảo không file nào bị lỗi
ckpt_dir = OUTPUT_ROOT / "checkpoints" / "tv3_week2"
ckpts = sorted(list(ckpt_dir.glob("*/best.pt")))
if not ckpts:
    raise RuntimeError("Chưa có checkpoint nào được tạo! KHÔNG ĐƯỢC ngắt runtime.")

all_valid = True
for cp in ckpts:
    try:
        meta = torch.load(cp, map_location="cpu", weights_only=True)
        assert "model" in meta, "Thiếu key 'model' trong checkpoint"
        best_ep = meta.get("best_epoch", meta.get("epoch", "?"))
        best_dice = meta.get("best_dice", meta.get("val_dice", 0.0))
        print(f"  [OK - TOÀN VẸN] {cp.parent.name}/best.pt | Best Epoch: {best_ep} | Best Val Dice: {best_dice:.4f}")
    except Exception as exc:
        print(f"  [LỖI - CORRUPTED] {cp}: {exc}")
        all_valid = False

if not all_valid:
    raise RuntimeError("PHÁT HIỆN CHECKPOINT BỊ LỖI! Dừng ngay để giữ nguyên runtime cho bạn kiểm tra.")

print("\n>>> KHÓA AN TOÀN ĐẠT CHUẨN! Toàn bộ trọng số đã nằm an toàn trên Google Drive.")
print(">>> Sẵn sàng chuyển sang Cell ngắt kết nối runtime.")

In [ ]:
# 8. Tự động ngắt kết nối Colab khi hoàn tất (chỉ chạy sau khi Cell 7 xác nhận an toàn tuyệt đối)
from google.colab import runtime
runtime.unassign()